# 05 · Complete-corpus multilingual OCR with OpenEnv

Use all **1,006,170 Nayana pages across 22 languages** through one indexed environment. Source Parquets live in the [HuggingEnvs bucket](https://huggingface.co/buckets/HuggingEnvs/NayanaOCR_Corpus_2025_bucket). Metadata lookup is separate from image loading; training consumes shuffled source blocks with bounded prefetch and repeats pinned task IDs for GRPO.

This notebook calls the package and shared runner. Start from `05-multilingual-ocr/`:

```bash
uv run --frozen --project envs/nayana_ocr --extra train --with jupyterlab jupyter lab
```

The default uses the hosted Space and its complete index. Set `START_LOCAL=True` to run the same server locally. A cold image block usually fetches roughly 100 JPEGs (70–110 MB); this short walkthrough can warm two blocks. Metadata queries never fetch page images. GPU training is optional and disabled by default.

Source: [CognitiveLab / NayanaOCR_Corpus_2025](https://huggingface.co/datasets/Cognitive-Lab/NayanaOCR_Corpus_2025), CC BY-NC 4.0. Copied data, annotations, indexes and images retain this attribution/license. Source rendering and annotation quality still need auditing before a model benchmark.

In [ ]:
import json
from contextlib import ExitStack
from pathlib import Path

PROJECT = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "05-multilingual-ocr")
    if candidate.name == "05-multilingual-ocr" and (candidate / "project.yaml").exists()
)
MANIFEST = PROJECT / "data" / "corpus-manifest.json"
START_LOCAL = False
SOURCE_ROOT = None  # For an existing bucket mount, use "/corpus".
RUN_GPU_TRAINING = False
URL = "https://huggingenvs-nayana-ocr-env.hf.space"
stack = ExitStack()
pinned = json.loads(MANIFEST.read_text())
print(
    {
        "snapshot": pinned["snapshot_id"],
        "pages": sum(pinned["pages"].values()),
        "languages": len(pinned["pages"]),
    }
)

### Layout and strict descriptive VQA
The full index now includes `layout_detection` (six source labels and pixel-coordinate JSON boxes) and `descriptive_vqa` (strict Gemma reference-based grading). Both use the same indexed access and prefetch pipeline. Layout reward averages class-aware region F1 over IoU 0.50–0.95. Descriptive VQA gives reward 1 only when all six judge checks pass.
The hosted Space has its judge configured. A local or colocated HF Jobs server needs `NAYANA_JUDGE_TOKEN`/`HF_TOKEN` with Inference Providers permission. The served manifest records the judge model/provider and policy hash, and training saves that manifest. Provider failures produce errors, never synthetic zero rewards. See [JUDGE.md](../JUDGE.md) for deployment and calibration.


## Open the same full index locally or on the Space

The manifest pins source identity and per-language SQLite checksums. Indexes are copied to local disk lazily; mounted SQLite files are not queried remotely. Local image reads use HTTP ranges by default, or the existing mount supplied below. To reproduce the corpus copy/index itself, follow [REPRODUCE.md](../REPRODUCE.md); no 813 GB workstation download is required.

In [ ]:
from nayana_ocr.client import connect
from nayana_ocr.corpus_training import BlockTaskStream, CorpusAPI
from nayana_ocr.runtime import local_server

if START_LOCAL:
    URL = stack.enter_context(
        local_server(
            MANIFEST,
            web=True,
            source_root=SOURCE_ROOT,
            cache_dir=PROJECT / "data" / "corpus-cache-notebook",
        )
    )
api = CorpusAPI(URL, pinned["snapshot_id"])
stack.callback(api.close)
print("Playground:", URL + "/web/")
print("Indexed tasks:", sum(row["tasks"] for row in api.manifest["counts"]))
print("Languages:", api.manifest["config"]["languages"])

## Jump to the end of the corpus without opening an image

OpenEnv's existing TaskProvider API supplies counts and indexed lookup. A task ID includes the immutable snapshot, language, split, and position. Document-level hash partitions keep translations, crops and questions from a document in the same train/validation/test split. Metadata has no reference answers; the asset is materialized only on reset.

In [ ]:
with connect(URL) as client:
    total = client.num_tasks("train")
    last = client.get_task_range("train", total - 1, total)[0]
assert "reference" not in last and last["asset_path"] is None
print(
    {key: last[key] for key in ("task_id", "language", "family", "page_id", "block_id")}
)

## Iterate source blocks with bounded prefetch

The finite iterator hash-shuffles row groups, then shuffles 128-record metadata chunks within each group. Every selected task appears once per epoch; prefetch changes cache timing, not the sequence. The full GRPO recipe uses this iterator across epochs and lets TRL own G-fold completion repetition. Select any subset of the 22 languages and five task families. The default full pass preserves natural task proportions; fixed evaluation IDs are selected using a separate balanced indexed sample.

In [ ]:
from itertools import islice

settings = dict(languages=["en"], families=["page_ocr"], seed=42, prefetch_blocks=2)
stream = BlockTaskStream(api, **settings)
iterator = iter(stream)
rows = list(islice(iterator, 8))
assert len({row["task_id"] for row in rows}) == 8
print(
    {
        "blocks": len(stream.plan),
        "tasks_per_epoch": sum(b["tasks"] for b in stream.plan),
        "first_tasks": [row["task_id"] for row in rows[:2]],
    }
)
state = stream.state_dict()
resumed = BlockTaskStream(api, **settings)
resumed.load_state_dict(state)
assert next(iter(resumed))["task_id"] == next(iterator)["task_id"]
print("Consumer cursor replay matched.")

## Replay one task in two independent RL sessions

Reset resolves the selected ID, loads its group once, and caches the rendered full-page image. The client fetches hash-verified binary media and shares it across repeated completions. The full-page task masks unannotated areas and scores annotated text in geometric reading order. Empty answers score zero. The playground separately reveals reference text after scoring.

A source group is usually 100 pages, so arbitrary cold access is more expensive than reusing a warm block. `/data/cache` reports prefetch, load, eviction and byte counters. Mounted network traffic is not included in the application's HTTP byte counter.

In [ ]:
import requests
from nayana_ocr.training import AssetCache, TrainingEnvironment, env_reward

cache = AssetCache(URL)
envs = [TrainingEnvironment(URL, cache, api.snapshot_id) for _ in range(2)]
try:
    images = [env.reset(task_id=rows[0]["task_id"])[0]["image"] for env in envs]
    assert images[0].size == images[1].size and cache.downloads == 1
    rewards = env_reward(["", ""], envs, [rows[0]["task_id"]] * 2)
    assert rewards == [0.0, 0.0]
    print(
        {
            "dimensions": images[0].size,
            "client_media_downloads": cache.downloads,
            "rewards": rewards,
        }
    )
    for image in images:
        image.close()
finally:
    for env in envs:
        env._close()
print(requests.get(URL + "/data/cache", timeout=180).json())

## Optional single-GPU GRPO

This calls the shared runner, including baseline evaluation, LoRA GRPO, adapter-update/finite-loss checks, and post-training evaluation. Qwen3-VL-2B is the default; two steps with G=2 are only an optimizer smoke. No GPU result or reward improvement is claimed by this CPU walkthrough. Consumer cursor replay above does not establish optimizer checkpoint replay: corpus/iterable optimizer resume is rejected until verified.

For HF Jobs, `train/hf_job.py` fetches a pinned Git commit and uses its lockfile. Attach `hf://buckets/HuggingEnvs/NayanaOCR_Corpus_2025_bucket:/corpus:ro` and pass `--corpus-manifest <published-manifest-uri> --source-root /corpus` to colocate training and serving. See [REPRODUCE.md](../REPRODUCE.md) for full commands, cache budgets, source-quality limits, and exact defaults.

In [ ]:
import subprocess
import sys

if RUN_GPU_TRAINING:
    subprocess.run(
        [
            sys.executable,
            str(PROJECT / "train" / "grpo_nayana.py"),
            "--env-url",
            URL,
            "--task-input",
            "corpus",
            "--prefetch-blocks",
            "2",
            "--smoke",
            "--output-dir",
            str(PROJECT / "results" / "local-notebook-grpo"),
        ],
        check=True,
    )
else:
    print("GPU training is disabled; serving and replay checks completed on CPU.")
stack.close()